<a href="https://colab.research.google.com/github/aaryanML/Pytorch/blob/main/Autograd_in_Pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

In PyTorch, **Autograd** is the automatic differentiation engine that powers neural network training.

It automatically computes gradients (derivatives) for tensors, which are needed for backpropagation in deep learning.

## Autograd on Scaler Tensors

### 1. Single Operation

In [ ]:
x = torch.tensor(3.0,requires_grad=True)
# requires_grad tells pytorch to track operations on x

In [ ]:
y = x**2
# Pytorch builds a computation graph => x -> square -> y

In [ ]:
y

tensor(9., grad_fn=<PowBackward0>)

In [ ]:
y.backward()
# y.backward() computes derivative of y wrto x

In [ ]:
x.grad
# x.grad stores the gradient dy/dx

tensor(6.)

y = x**2\
dy/dx = 2x\
dy/dx at x=3 => 2 * 3 = 6

### 2. Multiple Operations

In [ ]:
x = torch.tensor(4.0,requires_grad=True)

In [ ]:
y = x**2

In [ ]:
z = torch.sin(y)

In [ ]:
x

tensor(4., requires_grad=True)

In [ ]:
y

tensor(16., grad_fn=<PowBackward0>)

In [ ]:
z

tensor(-0.2879, grad_fn=<SinBackward0>)

In [ ]:
z.backward()

In [ ]:
x.grad

tensor(-7.6613)

In [ ]:
y.grad

/tmp/ipykernel_2408/486760323.py:1: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:494.)
  y.grad


The warning appears because y is a non-leaf tensor\
PyTorch computes its gradient during backpropagation but does not save it in  **y.grad**  unless you call  **y.retain_grad()**  beforehand.

## Autograd on Vector Tensors

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0],requires_grad=True)

In [ ]:
y = (x**2).mean()

In [ ]:
x

tensor([1., 2., 3.], requires_grad=True)

In [ ]:
y

tensor(4.6667, grad_fn=<MeanBackward0>)

In [ ]:
y.backward()

In [ ]:
x.grad

tensor([0.6667, 1.3333, 2.0000])

y = (x_1 ** 2 + x_2 ** 2 + x_3 ** 2) / 3\
dy/dx_1 = (2 * x1)/3\
dy/dx_2 = (2 * x2)/3\
dy/dx_3 = (2 * x3)/3

## Why do we need to clear gradients?

In PyTorch, gradients accumulate by default.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)

y = x**2
y.backward()

print(x.grad)   # 4

y = x**2
y.backward()

print(x.grad)   # 8, not 4!

tensor(4.)
tensor(8.)


After the second .backward(), PyTorch adds the new gradient to the existing one:

                            x.grad = 4+4 = 8

This behaviour is useful for :

Gradient accumulation across multiple mini-batches.\
Computing gradients from multiple losses.

But during normal training, we usually want gradients from only the current batch, so we must clear old gradients first.

In [ ]:
# x.grad.zero_() -> clears gradient | _ for permamnent changes in x

x = torch.tensor(2.0, requires_grad=True)

y = x**2
y.backward()

print(x.grad)
x.grad.zero_()

y = x**2
y.backward()

print(x.grad)

tensor(4.)
tensor(4.)


## Disable Gradient Tracking

Gradient tracking is needed only when you're training a model.

During :

  * Inference (making predictions)
  * Evaluation/validation
  * Data preprocessing
  * Any computation where gradients are not needed

tracking gradients wastes:

  * Memory (stores the computation graph)
  * Computation time (records every operation)

So we disable it to make code faster and more memory-efficient.

In [ ]:
# torch.no_grad()

x = torch.tensor([1., 2., 3.], requires_grad=True)

with torch.no_grad():
    y = x * 2

print(y.requires_grad)

False


In [ ]:
y.backward()

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [ ]:
# detach() => If you want a tensor that is disconnected from the graph

y = x * 2

z = y.detach()

In [ ]:
z.requires_grad

# z shares the same data as y but is not connected to the computation graph.

False

In [ ]:
# torch.inference_mode() => even faster for inference
# Compared to no_grad():
    # Uses less overhead
    # Intended specifically for inference
    # Usually gives the best performance for prediction-only workloads


with torch.inference_mode():
    predictions = model(X)